In [9]:
import os
from datetime import datetime
import pandas as pd
import numpy as np
import sys
import csv
import openpyxl

In [10]:
df = pd.read_excel(r"C:\Users\USER\Desktop\K1.xlsx")


In [11]:
if "1=SP,2=WH" in df.columns:
    # เช็กว่ามีแถวไหนที่มีค่าเท่ากับ 0 ไหม
    if (df["1=SP,2=WH"] == 0).any():
        print(
            "❌ [ERROR]: ตรวจพบค่า '0' ในคอลัมน์ 1=SP,2=WH ซึ่งไม่ถูกต้อง! ระบบหยุดทำงานชั่วคราว กรุณาตรวจเช็กไฟล์ Excel"
        )
        sys.exit()  # สั่งให้โปรแกรมหยุดทำงานทันที ไม่ให้ไปเซฟไฟล์ผิดๆ

In [12]:
if "RECEIVE_PIECE" in df.columns:
    df.loc[df["RECEIVE_PIECE"] == 0, "ReBplus"] = np.nan
    print("🧹 เคลียร์ค่าว่างสำหรับแถวที่ RECEIVE_PIECE == 0 เรียบร้อยแล้ว")

🧹 เคลียร์ค่าว่างสำหรับแถวที่ RECEIVE_PIECE == 0 เรียบร้อยแล้ว


In [13]:


# ==========================================
# 1. ระบบดึงวัน-เดือน-ปี ปัจจุบัน (เป็น พ.ศ. อัตโนมัติ)
# ==========================================
now = datetime.now()
thai_year = (
    now.year + 543 - 2500
)  # แปลงปี ค.ศ. เป็น พ.ศ. และเอาแค่ 2 หลักท้าย (เช่น 2026 + 543 = 2569 -> 69)

# จะได้รูปแบบ "วัน-เดือน-ปี รับ" เช่น "17-5-69 รับ" (อิงตามวันที่รันโค้ดจริง)
DATE_SUFFIX = f"{now.day}-{now.month}-{thai_year} รับ"


# ==========================================
# 2. ระบุที่อยู่โฟลเดอร์ปลายทาง (เหมือนเดิม)
# ==========================================
# เปลี่ยน XXXXXXXXXXXXXXXXXXXX เป็น Path จริงของคุณ
PATHS = {
    "11": r"C:\Users\USER\Desktop\11",
    "11_00": r"C:\Users\USER\Desktop\1100",
    "21": r"C:\Users\USER\Desktop\21",
    "21_00": r"C:\Users\USER\Desktop\2100",
    "31": r"C:\Users\USER\Desktop\31",
    "31_00": r"C:\Users\USER\Desktop\3100",
    "41": r"C:\Users\USER\Desktop\41",
    "41_00": r"C:\Users\USER\Desktop\4100",
    "51": r"C:\Users\USER\Desktop\51",
    "51_00": r"C:\Users\USER\Desktop\5100",
    "SP": r"C:\Users\USER\Desktop\SP00"
}

# กำหนดชื่อย่อสาขา
BRANCH_NAMES = {'00': 'WH',"11": "K1", "21": "K2", "31": "K3", "41": "K4", "51": "K5"}

In [14]:
def detect_file_branch(df):
    # ดึงรหัสตำแหน่งที่ 2 ทั้งหมดที่มีอยู่ในไฟล์นี้ออกมาดู (เอาเฉพาะค่าที่ไม่ซ้ำกัน และล้างช่องว่าง)
    all_codes = df["ReBplus"].astype(str).str.split(",").str.get(2).str.strip().unique()
    
    # วิ่งเช็กหารหัสสาขาหลักก่อน
    for branch_code in ["11", "21", "31", "41", "51"]:
        if branch_code in all_codes:
            return branch_code  # เจอเลขสาขาหลักปนอยู่ สรุปว่าเป็นของสาขานั้นทันที
            
    # ถ้าในไฟล์ไม่มีรหัสสาขาหลักเลย และมีรหัส "00" อยู่ แปลว่าเป็นคลังใหญ่ SP
    if "00" in all_codes:
        return "SP"
        
    return None

In [15]:
# --- เริ่มทำงานคัดกรองข้อมูล ---NEW
# ==============================================================================
current_branch = detect_file_branch(df)

if current_branch is not None:
    # 1. เตรียมข้อมูลสปลิตเพื่อดูรหัสตรงกลางจาก ReBplus
    sku_str = df["ReBplus"].astype(str).str.strip()
    extracted_code = sku_str.str.split(",").str.get(2).str.strip()
    
    # 2. 🟢 ดึงข้อมูลผ่าน "ชื่อคอลัมน์ตรงๆ" เพื่อให้มั่นใจว่าไม่สลับช่อง (ต่อให้ทำ Table ใน Excel มาก็ตาม)
    # หากหาชื่อคอลัมน์ตรงๆ ไม่เจอ ระบบจะไปดึงคอลัมน์ช่องที่ 2 มาแทนเพื่อความปลอดภัย
    type_col_name = "1=SP,2=WH"
    if type_col_name in df.columns:
        col_type = df[type_col_name]
    else:
        # ดักจับกรณีชื่อสะกดเพี้ยน มีเว้นวรรค เช่น "1=SP, 2=WH"
        alt_name = [c for c in df.columns if "1=" in str(c) or "WH" in str(c)]
        col_type = df[alt_name[0]] if alt_name else df.iloc[:, 1]

    # จัดการชื่อเรียกสาขา
    b_name = BRANCH_NAMES.get(current_branch, current_branch)
    print(f"🎯 ตรวจพบข้อมูลไฟล์ของ: {b_name} (รหัสหลัก {current_branch})")


    # ==========================================================================
    # 🏪 [ส่วนที่ 1] กรองผ่านคอลัมน์ประเภท == 1 (หน้าร้าน)
    # ==========================================================================
    if current_branch == "SP":
        cond_main = (extracted_code == "00")
    else:
        # 🟢 เงื่อนไขตรงตามที่คุณกำหนด: คอลัมน์ประเภทต้องระบุกลุ่มเลข 1 และรหัสใน ReBplus ต้องตรงกับสาขา
        cond_main = col_type.isin([1, 1.0, "1", "1.0"]) & (extracted_code == current_branch)
        
    main_data = df.loc[cond_main, "ReBplus"]

    if not main_data.empty:
        main_file_name = f"{b_name}-SP-{DATE_SUFFIX}.txt"
        main_full_path = os.path.join(PATHS[current_branch], main_file_name) 
        
        with open(main_full_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(main_data.astype(str).tolist()))
        print(f"   -> 🏪 เซฟหน้าร้านสำเร็จ: {main_full_path} (รวม {len(main_data)} แถว)")


    # ==========================================================================
    # 🏢 [ส่วนที่ 2] กรองผ่านคอลัมน์ประเภท == 2 (โกดัง)
    # ==========================================================================
    if current_branch == "SP":
        data_00 = [] 
    else:
        # 🟢 เงื่อนไขตรงตามที่คุณกำหนด: คอลัมน์ประเภทต้องระบุกลุ่มเลข 2 (หรือค่าว่างจาก Excel) และรหัสใน ReBplus ต้องเป็น "00"
        cond_00 = (col_type.isin([2, 2.0, "2", "2.0"]) | col_type.isna() | (col_type.astype(str).str.strip() == "nan")) & (extracted_code == "00")
        data_00 = df.loc[cond_00, "ReBplus"]

    if not (isinstance(data_00, list) or data_00.empty):
        file_name_00 = f"{b_name}-WH-{DATE_SUFFIX}.txt"
        key_00 = f"{current_branch}_00"  
        full_path_00 = os.path.join(PATHS[key_00], file_name_00)
        
        with open(full_path_00, 'w', encoding='utf-8') as f:
            f.write('\n'.join(data_00.astype(str).tolist()))
        print(f"   -> 🏢 เซฟโกดังสำเร็จ  : {full_path_00} (รวม {len(data_00)} แถว)")
    else:
        if current_branch != "SP":
            print(f"   ⚠️ ไม่ยอมเซฟไฟล์โกดัง! (ตรวจสอบแล้วไม่มีแถวไหนที่ 'คอลัมน์ประเภทเป็นเลข 2' คู่กับ 'รหัสกลางเป็น 00')")

else:
    print("❌ ไม่พบโครงสร้างรหัสสาขาที่ถูกต้องในไฟล์นี้  ไม่สามารถประมวลผลได้")

🎯 ตรวจพบข้อมูลไฟล์ของ: K1 (รหัสหลัก 11)
   -> 🏪 เซฟหน้าร้านสำเร็จ: C:\Users\USER\Desktop\11\K1-SP-18-5-69 รับ.txt (รวม 287 แถว)
   -> 🏢 เซฟโกดังสำเร็จ  : C:\Users\USER\Desktop\1100\K1-WH-18-5-69 รับ.txt (รวม 144 แถว)
